# 40 Presubmission robustness: matched non-fragment controls and generalization audit

This notebook evaluates provenance/confounding, practical-effect, toxicity-label and multiplicity concerns identified during expert presubmission review. It is a **post-freeze sensitivity analysis**. It does not alter the frozen folds, selected pipelines or external cohorts from Notebooks 01–39.

The primary new comparison is the frozen final MS/MS model versus a flexible non-fragment control containing the complete non-fragment input set available to the Transformer: eight numeric metadata fields plus adduct, instrument category and source library.

Expected runtime is machine-dependent. The fixed HGB controls are CPU models; the original neural models are not retrained.

In [ ]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / 'outputs' / 'pbt_msms_v1').exists():
    ROOT = ROOT.parent
if not (ROOT / 'outputs' / 'pbt_msms_v1').exists():
    ROOT = Path(r'E:\谱图预测PBT_终版')
assert (ROOT / 'outputs' / 'pbt_msms_v1').exists(), 'Project root not found.'

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pbt_msms.presubmission_model_audit as analysis_module
analysis_module = importlib.reload(analysis_module)
print('ROOT:', ROOT)
print('Python:', sys.executable)
print('Module version:', analysis_module.VERSION)

## Fixed analysis plan

Review this cell before execution. The plan is written to the new output directory and cannot silently change on a later rerun.

In [ ]:
PLAN = analysis_module.analysis_plan()
display(pd.Series(PLAN, name='fixed_value').to_frame())
print(json.dumps(PLAN['model_policy'], ensure_ascii=False, indent=2))

## Input validation

This checks every frozen upstream artifact and the Transformer metadata schema without writing analysis results.

In [ ]:
input_check = analysis_module.validate_inputs(ROOT)
display(input_check)
assert input_check['exists'].all()
print('PASS: all required frozen inputs are available.')

## Execute

`N_BOOTSTRAP=2000` is the manuscript setting. For a quick code test only, you may temporarily use 100, but rerun with 2000 before using any result in the paper. External labels are joined only after the new external predictions are frozen without labels.

In [ ]:
N_BOOTSTRAP = 2000
SELECTION_BOOTSTRAP = 2000
RUN_FULL_ANALYSIS = False  # Set True only when intentionally refitting all nonfragment controls.
p = analysis_module.paths(ROOT)
if RUN_FULL_ANALYSIS:
    result = analysis_module.run_analysis(ROOT, n_bootstrap=N_BOOTSTRAP, selection_bootstrap=SELECTION_BOOTSTRAP)
else:
    # Recorded refresh reused frozen controls and external predictions after input-hash checks.
    result = {'manifest': json.loads(p['manifest'].read_text(encoding='utf-8')),
              'summary': p['summary'], 'figure': p['figure_png'], 'output_dir': p['out']}
    for name, key in {'paired':'paired', 'source_strata':'source_strata',
        'similarity_strata':'similarity_strata', 'toxicity_strata':'toxicity_strata',
        'effect_size_context':'effect_context', 'utility':'utility',
        'selection_stability':'selection_stability', 'qc':'qc'}.items():
        result[name] = pd.read_csv(p[key])
display(pd.Series(result['manifest'], name='value').to_frame())


## Primary matched-control results

In [ ]:
primary = result['paired'].loc[
    result['paired']['candidate'].eq(analysis_module.PIPELINE_FINAL)
    & result['paired']['reference'].eq(analysis_module.PRIMARY_MATCHED_ROUTE)
    & result['paired']['cohort'].isin(['development_oof', 'novel_all', 'transfer'])
].copy()
display(primary[[
    'cohort', 'task', 'n', 'metric', 'observed_improvement',
    'ci_low', 'ci_high', 'one_sided_p', 'holm_p', 'bh_q'
]])

## Provenance and structural-similarity strata

In [ ]:
display(result['source_strata'])
display(result['similarity_strata'])

## Toxicity species-duration sensitivity

In [ ]:
display(result['toxicity_strata'])
display(result['effect_size_context'])

## Screening utility and model-selection stability

In [ ]:
display(result['utility'])
display(result['selection_stability'].sort_values(['task', 'bootstrap_selection_probability'], ascending=[True, False]))

## Quality control and outputs

In [ ]:
display(result['qc'])
assert result['qc']['passed'].astype(bool).all()
print('Summary:', result['summary'])
print('Figure:', result['figure'])
print('Output directory:', result['output_dir'])